# 03 일괄검출 및 동영상분석

여러 이미지에 대한 일괄 객체검출과 동영상 기반 객체검출·추적을 수행합니다.

> ⚠️ **동영상 파일 주의**
> - `.wmv` 파일은 Google Colab(Linux)에서 재생되지 않습니다.
> - Colab 수강생은 `.mp4` 파일을 사용하거나, 아래 셀에서 공개 샘플을 다운로드하세요.

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
from ultralytics import YOLO

# ── 경로 설정 ──────────────────────────────────────────────
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent

IMAGE_DIR  = ROOT / 'data' / 'images'
VIDEO_DIR  = ROOT / 'data' / 'videos'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
video_exts = {'.mp4', '.avi', '.mov', '.wmv', '.mkv'}

images = sorted([p for p in IMAGE_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in image_exts]) \
         if IMAGE_DIR.exists() else []
videos = sorted([p for p in VIDEO_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in video_exts]) \
         if VIDEO_DIR.exists() else []

print('이미지 수:', len(images))
print('동영상 수:', len(videos))

model = YOLO('yolov8n.pt')

## 이미지 폴더 일괄 객체검출

In [ ]:
if not images:
    raise FileNotFoundError(f'이미지 없음 → {IMAGE_DIR} 에 파일을 넣어주세요.')

results = model.predict(
    source=str(IMAGE_DIR),
    conf=0.25,
    save=True,
    project=str(OUTPUT_DIR),
    name='detect_batch_nb',
    exist_ok=True,
)
print('처리 이미지 수:', len(results))

In [ ]:
rows = []
counter = Counter()
for result in results:
    rows.append({'image': Path(result.path).name, 'detect_count': len(result.boxes)})
    for class_id in result.boxes.cls.tolist():
        counter[result.names[int(class_id)]] += 1

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print('\n클래스별 검출 수')
for key, value in sorted(counter.items()):
    print(f'  - {key}: {value}')

## 동영상 객체검출 (predict 모드)

In [ ]:
if not videos:
    raise FileNotFoundError(
        f'동영상 없음 → {VIDEO_DIR} 에 .mp4 파일을 넣어주세요.\n'
        '  Google Colab 수강생: 아래 명령으로 샘플 다운로드\n'
        '  !wget -q https://ultralytics.com/assets/decelera_landscape_min.mov -O data/videos/sample.mov'
    )

video_path = videos[0]
if video_path.suffix.lower() == '.wmv':
    print('⚠️  WMV 파일 — Google Colab에서는 재생되지 않습니다.')

model.predict(
    source=str(video_path),
    conf=0.3,
    save=True,
    project=str(OUTPUT_DIR),
    name='video_detect_nb',
    exist_ok=True,
)
print('동영상 검출 완료:', OUTPUT_DIR / 'video_detect_nb')

## 동영상 객체추적 (track 모드)

- `model.track()` → 동일 객체에 고유 ID 부여, 프레임 간 연속성 유지
- 결과 영상에서 박스 위 **ID: 1, ID: 2...** 숫자가 표시됨

In [ ]:
model.track(
    source=str(video_path),
    conf=0.3,
    save=True,
    project=str(OUTPUT_DIR),
    name='video_track_nb',
    exist_ok=True,
)
print('추적 완료:', OUTPUT_DIR / 'video_track_nb')

In [ ]:
# 결과 요약 저장
summary = df.copy()
summary['notes'] = ''
csv_path = OUTPUT_DIR / 'result_summary_nb.csv'
summary.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('저장 완료:', csv_path)